In [1]:
from functions.PPTX_CHARTER_KT_THEME import * 

## Preparación del dataset

In [2]:
# Carga de datasets
data_saturation = pd.read_excel('Input.xlsx', sheet_name='Saturation')
data_raw = pd.read_excel('Input.xlsx', sheet_name='Raw_data', index_col='Period')
data_contrib_vol = pd.read_excel('Input.xlsx', sheet_name='Contribution_volume', index_col='Period')
data_contrib_rev = pd.read_excel('Input.xlsx', sheet_name='Contribution_revenue', index_col='Period')
data_breaks = pd.read_excel('Input.xlsx', sheet_name='Breaks')

# Identificación de origen de datos a través de nombres de columnas
data_raw.columns = pd.Series(data_raw.columns).apply(lambda x: f'{x}_Input')
data_contrib_rev.columns = pd.Series(data_contrib_rev.columns).apply(lambda x: f'{x}_Contribution_Rev')
data_contrib_vol.columns = pd.Series(data_contrib_vol.columns).apply(lambda x: f'{x}_Contribution_Vol')

# Agregación en un solo dataframe 
data_total = pd.merge(
    left=data_raw,
    right=data_contrib_rev,
    left_index=True,
    right_index=True
)

data_total = pd.merge(
    left=data_total,
    right=data_contrib_vol,
    left_index=True,
    right_index=True
)

# Construcción de tabla de jerarquía 
heirarchy = pd.DataFrame({'variable_name': data_total.columns})

heirarchy[['L0','L1','L2','L3','L4','Metric Type','Metric','Aggregation']] = pd.NA

# Añade pestaña para construir jerarquía 
with pd.ExcelWriter('Input.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    heirarchy.to_excel(writer, sheet_name='Heirarchy-CONSTRUCTOR')

# Fix de nombres de curvas de saturación a L2 L3 L4

name_map = {
    'FB_YT':'Media Digital YT+Meta',
    'Google Paid':'Media Search Google',
    'Online Display':'Media Digital Display',
    'Out of Home Retail':'Media OOH Retail',
    'Out of Home Screen':'Media OOH Screen'
}

data_saturation['Variable'] = data_saturation['Variable'].map(name_map)


In [3]:
# Lee la jerarquía final 
heirarchy = pd.read_excel('Input.xlsx', sheet_name='Heirarchy', index_col=0)

In [4]:
# Apilación del dataset total 
data_stacked = data_total.melt(ignore_index=False)
data_stacked.reset_index(inplace=True)
data_stacked = data_stacked.replace(0,pd.NA)

# Unión con jerarquía
data_stacked = data_stacked.merge(
    right=heirarchy,
    left_on='variable',
    right_on='variable_name', 
    how='left'
)

# Unión con breaks
data_stacked = data_stacked.merge(
    data_breaks,
    left_on='Period',
    right_on='Period',
    how='left'
)

# Limpieza de NAs 
# data_stacked.dropna(inplace=True)

data_stacked['Period_str'] = data_stacked['Period'].dt.strftime('%Y-%m') # Formato para modelos mensuales '%Y-%m' || Formato para modelos semanales '%Y-%m-%d'

# Eliminación de variables temporales 
del data_raw, data_contrib_vol, data_contrib_rev, data_total, heirarchy, data_breaks, name_map

## Reporte

In [5]:
# Mapeo de labels customizados para identificar variables relevantes en la construcción del reporte 
config = {
    'L1 Label de marca': ' Twinings',
    'L1 Label de categoría': 'Category',
    'L0 Label de ventas': 'Sales',
    'L0 Label de distribución': 'Distribution',
    'L0 Label de descuento': 'Discounts',
    'L0 Label de precio': 'Price',
    'L0 Label de macroeconómicos': 'External',
    'Metric Label de ventas en volumen': 'kg',
    'L0 Label media organica': 'Organic Media'
}

In [6]:
# Opcional: Añadir el color de la marca del cliente a la paleta 
original_pallete = [color for color in standard_pallete]
client_color = RGBColor(51,51,51)
standard_pallete.insert(0, client_color)

In [7]:
# Inicialización de la presentación y selección de slide layouts 
prs = Presentation('./pptx_template/Kantar Master Template - LIFTROI ENG.pptx')

layout_section_header_white = prs.slide_layouts[4]
layout_blank = prs.slide_layouts[22]

### Sección 1: Contribuciones

In [8]:
# Slide de contribuciones periodo a periodo por primer nivel de agregación 

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=False,
    slide_title='Overall sales volume contribution',
    stacked_df=data_stacked,
    value_var='value',
    category_vars=['Breaks'],
    series_vars=['L2'],
    filter_mask=(data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']==config['Metric Label de ventas en volumen']) & (data_stacked['Breaks'].notna()),
    chart_type=XL_CHART_TYPE.COLUMN_STACKED,
    show_legend=True,
    show_category_axis=True,
    show_data_labels_perc=True,
    color_pallete=original_pallete,
    zero_axis_start=False
)

c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [9]:
# Zoom de inversión en medios en nivel 2

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=False,
    slide_title='Media sales volume contribution',
    stacked_df=data_stacked,
    value_var='value',
    category_vars=['Breaks'],
    series_vars=['L2','L3'],
    filter_mask=(data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']==config['Metric Label de ventas en volumen']) & (data_stacked['L0']=='Media') & (data_stacked['Breaks'].notna()),
    chart_type=XL_CHART_TYPE.COLUMN_STACKED,
    show_legend=True,
    show_category_axis=True,
    show_data_labels_perc=True,
    color_pallete=original_pallete
)

c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [10]:
# Zoom de inversión en medios en nivel 3

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=False,
    slide_title='Media sales volume contribution',
    stacked_df=data_stacked,
    value_var='value',
    category_vars=['Breaks'],
    series_vars=['L2','L3','L4'],
    filter_mask=(data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']==config['Metric Label de ventas en volumen']) & (data_stacked['L0']=='Media')& (data_stacked['Breaks'].notna()),
    chart_type=XL_CHART_TYPE.COLUMN_STACKED,
    show_legend=True,
    show_category_axis=True,
    show_data_labels_perc=True,
    color_pallete=original_pallete
)

c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [11]:
# Grafico a periodo, contribución total vs contribución de medios abierto por el nivel 2

slide_temp = prs.slides.add_slide(layout_blank)

slide_temp.placeholders[0].text = 'Contribución de medios vs contribución total en volumen ###PIMPEAR###'

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data_stacked,
    category_vars=['Period_str'],
    series_vars=['L3'],
    value_var='value',
    filter_mask=(data_stacked['L0']=='Media') & (data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']==config['Metric Label de ventas en volumen']),
    chart_type=XL_CHART_TYPE.AREA_STACKED,
    color_pallete=original_pallete
)

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data_stacked,
    category_vars=['Period_str'],
    series_vars=['Metric'],
    value_var='value',
    filter_mask=(data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']==config['Metric Label de ventas en volumen']),
    show_data_labels_perc=False,
    show_legend=False,
    chart_type=XL_CHART_TYPE.LINE
)

c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)
c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [12]:
# Slide de inversion en medios a la izquierda y contribución de medios a la derecha 

slide_temp = prs.slides.add_slide(layout_blank)

slide_temp.placeholders[0].text = 'Inversion en medios vs contribución en volumen'

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    slide_title='Inversión en medios pagados',
    stacked_df=data_stacked,
    category_vars=['Breaks'],
    series_vars=['L3'],
    value_var='value',
    filter_mask=(data_stacked['L0']=='Media') & (data_stacked['Metric Type']=='Spend')& (data_stacked['Breaks'].notna()),
    show_data_labels_perc=True,
    pos_horizontal=Inches(0.39),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37),
    color_pallete=original_pallete
)

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    slide_title='Media sales volume contribution',
    stacked_df=data_stacked,
    value_var='value',
    category_vars=['Breaks'],
    series_vars=['L2','L3'],
    filter_mask=(data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']==config['Metric Label de ventas en volumen']) & (data_stacked['L0']=='Media')& (data_stacked['Breaks'].notna()),
    chart_type=XL_CHART_TYPE.COLUMN_STACKED,
    show_legend=True,
    show_category_axis=True,
    show_data_labels_perc=True,
    pos_horizontal=Inches(6.78),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37),
    color_pallete=original_pallete
)

c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)
c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [13]:
# Slide con share of spend, share of contribution, idx spend / contribution y ROI

# Preparación del set de datos de contribución, spend y ROI

contribution_rev = get_crosstab(
    stacked_df=data_stacked,
    index_vars=['L2','L3','L4'],
    column_vars=['Breaks2'],
    value_var='value',
    filter_mask=(data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']=='$') & (data_stacked['L0']=='Media')
)

contribution_vol = get_crosstab(
    stacked_df=data_stacked,
    index_vars=['L2','L3','L4'],
    column_vars=['Breaks2'],
    value_var='value',
    filter_mask=(data_stacked['Metric Type'] == 'Contribution') & (data_stacked['Metric']==config['Metric Label de ventas en volumen']) & (data_stacked['L0']=='Media')
)

spend = get_crosstab(
    stacked_df=data_stacked,
    index_vars=['L2','L3','L4'],
    column_vars=['Breaks2'],
    value_var='value',
    filter_mask=(data_stacked['L0']=='Media') & (data_stacked['Metric Type']=='Spend')
)

spend_filtered = spend[pd.Series(spend.index).apply(lambda x: x in contribution_rev.index).to_list()]

roi = (contribution_rev / spend.replace(0, pd.NA)).fillna(0).replace(np.inf, 0)
roi = roi[pd.Series(roi.index).apply(lambda x: x in contribution_rev.index).to_list()]

contribution_spend_idx = ((contribution_rev / contribution_rev.sum().replace(0, pd.NA)) / (spend_filtered / spend_filtered.sum().replace(0, pd.NA)).replace(0, pd.NA)).replace(np.inf, 0)

activity = get_crosstab(
    stacked_df=data_stacked,
    index_vars=['L2','L3','L4'],
    column_vars=['Breaks2'],
    value_var='value',
    filter_mask=(data_stacked['L0']=='Media') & (data_stacked['Metric Type']=='Effort')
)

activity_filtered = activity[pd.Series(activity.index).apply(lambda x: x in contribution_rev.index).to_list()]

efectividad = (contribution_vol / activity.replace(0, pd.NA)).fillna(0).replace(np.inf, 0)
efectividad = efectividad[pd.Series(efectividad.index).apply(lambda x: x in contribution_rev.index).to_list()]

cost_per_act = (spend_filtered / activity_filtered.replace(0, pd.NA)).fillna(0).replace(np.inf, 0)

# Creación de los gráficos 

slide_temp = prs.slides.add_slide(layout_blank)

slide_temp.placeholders[0].text = 'Media performance analysis'

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    crosstab_override=(spend_filtered / spend_filtered.sum().replace(0, pd.NA) * 100).replace(np.inf,0).fillna(0).round(1),
    chart_type=XL_CHART_TYPE.BAR_CLUSTERED,
    show_legend=True,
    height=Inches(4.37),
    width=Inches(2.98),
    pos_horizontal=Inches(0.39),
    pos_vertical=Inches(1.87),
    color_pallete=original_pallete,
    number_format_series='#,##0.0',
    show_data_labels_value=XL_LABEL_POSITION.OUTSIDE_END
)

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    crosstab_override=(contribution_rev / contribution_rev.sum().replace(0, pd.NA) * 100).replace(np.inf,0).fillna(0).round(1),
    chart_type=XL_CHART_TYPE.BAR_CLUSTERED,
    show_legend=True,
    height=Inches(4.37),
    width=Inches(2.98),
    pos_horizontal=Inches(3.58),
    pos_vertical=Inches(1.87),
    color_pallete=original_pallete,
    number_format_series='#,##0.0',
    show_data_labels_value=XL_LABEL_POSITION.OUTSIDE_END
)

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    crosstab_override=(contribution_spend_idx*100).round(1),
    chart_type=XL_CHART_TYPE.BAR_CLUSTERED,
    show_legend=True,
    height=Inches(4.37),
    width=Inches(2.98),
    pos_horizontal=Inches(6.77),
    pos_vertical=Inches(1.87),
    color_pallete=original_pallete,
    number_format_series='#,##0.0',
    show_data_labels_value=XL_LABEL_POSITION.OUTSIDE_END
)

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    crosstab_override=roi.round(2),
    chart_type=XL_CHART_TYPE.BAR_CLUSTERED,
    show_legend=True,
    height=Inches(4.37),
    width=Inches(2.98),
    pos_horizontal=Inches(9.95),
    pos_vertical=Inches(1.87),
    color_pallete=original_pallete,
    number_format_series='#,##0.00',
    show_data_labels_value=XL_LABEL_POSITION.OUTSIDE_END
)



C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_34748\777087234.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  roi = (contribution_rev / spend.replace(0, pd.NA)).fillna(0).replace(np.inf, 0)
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_34748\777087234.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  roi = (contribution_rev / spend.replace(0, pd.NA)).fillna(0).replace(np.inf, 0)
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_34748\777087234.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in 

In [14]:
# Slide con ROIs por medio, un grafico de columnas por año 

stacked_roi = roi.melt(ignore_index=False).reset_index()
stacked_roi['metric'] = 'ROI'

chart_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=False,
    slide_title='Media ROI',
    stacked_df=stacked_roi,
    value_var='value',
    category_vars=['agg_index_var'],
    series_vars=['metric'],
    iterable_vars=['agg_column_var'],
    grouping_vars=['metric'],
    chart_type= XL_CHART_TYPE.COLUMN_CLUSTERED,
    show_data_labels_value= XL_LABEL_POSITION.OUTSIDE_END,
    number_format_series='#,##0.00',
    show_legend=False
)

In [15]:
# Tabla resumen del performance de la campaña 

campaign_results = pd.merge(
    left=contribution_vol.round(0),
    right=contribution_rev.round(0),
    left_index=True,
    right_index=True,
    suffixes=(' Volume',' Revenue')
)

roi.columns = pd.Series(roi.columns).apply(lambda x: f'{x} ROI')

campaign_results = pd.merge(
    left=campaign_results,
    right=roi.round(2),
    left_index=True,
    right_index=True
)

table_maker(
    press_object=prs,
    layout_object=layout_blank,
    slide_title='Campaing contribution and ROI',
    crosstab_override=campaign_results,
    decimal_places=2
)

In [16]:
# Mapa estrategico, contribución x ROI x spend por medios 

# Preparación del apilado de la contribución en volumen
scatter_contrib = contribution_vol.melt(ignore_index=False)
scatter_contrib.reset_index(inplace=True)
scatter_contrib['media_idx'] = scatter_contrib[['agg_column_var','agg_index_var']].agg(' '.join, axis=1)
scatter_contrib = scatter_contrib.drop(columns=['agg_column_var','agg_index_var'])
scatter_contrib.columns = pd.Series(scatter_contrib.columns).apply(lambda x: 'contribution' if x == 'value' else x)

# Preparación del apilado del retorno
scatter_roi = roi.melt(ignore_index=False)
scatter_roi.reset_index(inplace=True)
scatter_roi['media_idx'] = scatter_roi[['agg_column_var','agg_index_var']].agg(' '.join, axis=1)
scatter_roi = scatter_roi.drop(columns=['agg_column_var','agg_index_var'])
scatter_roi.columns = pd.Series(scatter_roi.columns).apply(lambda x: 'roi' if x == 'value' else x)
scatter_roi['media_idx'] = scatter_roi['media_idx'].apply(lambda x: x.replace(' ROI',''))

# Preparación del apilado del spend 
scatter_spend = spend_filtered.melt(ignore_index=False)
scatter_spend.reset_index(inplace=True)
scatter_spend['media_idx'] = scatter_spend[['agg_column_var','agg_index_var']].agg(' '.join, axis=1)
scatter_spend = scatter_spend.drop(columns=['agg_column_var','agg_index_var'])
scatter_spend.columns = pd.Series(scatter_spend.columns).apply(lambda x: 'spend' if x == 'value' else x)

# Union para dataset del scatterplot 
scatter_df = pd.merge(
    left=scatter_spend,
    right=scatter_roi,
    left_on='media_idx',
    right_on='media_idx'
)

scatter_df = pd.merge(
    left=scatter_df,
    right=scatter_contrib,
    left_on='media_idx',
    right_on='media_idx'
)

scatter_df = scatter_df.fillna(0)
scatter_df.set_index('media_idx', inplace=True)
scatter_df = scatter_df.replace(0,pd.NA).dropna(subset=['spend','roi'])

# Generación del gráfico 
scatter_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=False,
    slide_title='Full media comparison',
    crosstab_df=scatter_df,
    x_col='contribution',
    y_col='roi',
    size_col='spend',
    chart_type=XL_CHART_TYPE.BUBBLE,
    color_pallete=original_pallete,
    marker_style=XL_MARKER_STYLE.CIRCLE,
    axis_center_mode='median'
)

C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_34748\1797192920.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scatter_df = scatter_df.fillna(0)


### Sección 2: Curvas de saturación 

In [17]:
# Curvas de saturación por medio 

for i, media in enumerate(data_saturation['Variable'].unique()):
    
    filtered_df = data_saturation[data_saturation['Variable']==media]
    filtered_df.set_index(['Activity'], inplace=True)
    filtered_df = filtered_df[['Contribution','ROI']]
    filtered_df.sort_index(inplace=True)

    chart_maker(
        press_object=prs,
        layout_object=layout_blank,
        use_slide_object=False,
        slide_title=f'Saturation curve – {media}',
        crosstab_override=filtered_df,
        chart_type=XL_CHART_TYPE.LINE,
        show_legend=True,
        color_pallete=original_pallete
    ) 


In [18]:
# Tabla de thresholds y valor promedio actual 

s_curve_thresholds = data_saturation.dropna()
s_curve_thresholds = s_curve_thresholds.pivot(columns='Points', index='Variable', values='Activity')

media_act_mean = get_crosstab(
    stacked_df=data_stacked,
    index_vars=['L2','L3','L4'],
    column_vars=['Metric Type'],
    value_var='value',
    filter_mask=(data_stacked['L0']=='Media') & (data_stacked['Metric Type']=='Effort'),
    agg_function='mean'
)

media_act_mean.columns = ['Current Average']

s_curve_thresholds = s_curve_thresholds.merge(
    right=media_act_mean,
    how='left',
    left_index=True,
    right_index=True
)

table_maker(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=False,
    slide_title='Saturation curves thresholds',
    crosstab_override=s_curve_thresholds
)


In [19]:
data_stacked[['L2','L3','L4']].astype('str').agg(' '.join, axis=1)

0                             Sales Sales Sales
1                             Sales Sales Sales
2                             Sales Sales Sales
3                             Sales Sales Sales
4                             Sales Sales Sales
                         ...                   
2955    External Seasonality Independence Month
2956    External Seasonality Independence Month
2957    External Seasonality Independence Month
2958    External Seasonality Independence Month
2959    External Seasonality Independence Month
Length: 2960, dtype: object

In [20]:
s_curve_thresholds[s_curve_thresholds.index==data_saturation['Variable'].unique()[0]]

,Breakthrough,Full Saturation,Maximum ROI,Start of Saturation,Current Average
Variable,,,,,
Media Digital Display,42160.000012,257040.000073,170000.000048,190400.000054,1177647.333333


In [21]:
for media in data_saturation['Variable'].unique():

    slide_temp = prs.slides.add_slide(layout_blank)
    slide_temp.placeholders[0].text = f'Media optimization - {media}'

    # Gráfico de actividad 
    chart_maker(
        press_object=prs,
        layout_object=layout_blank,
        use_slide_object=True,
        slide_object=slide_temp,
        stacked_df=data_stacked,
        value_var='value',
        category_vars=['Period_str'],
        series_vars=['Metric'],
        filter_mask=(data_stacked[['L2','L3','L4']].astype('str').agg(' '.join, axis=1)==media) & (data_stacked['Metric Type']=='Effort'),
        chart_type=XL_CHART_TYPE.COLUMN_CLUSTERED,
        show_legend=False,
        pos_horizontal=Inches(0.39),
        pos_vertical=Inches(1.87),
        width=Inches(6.15),
        height=Inches(2.94),
        color_pallete=original_pallete
    )

    filtered_thresholds = s_curve_thresholds[s_curve_thresholds.index==media]

    # Tabla con thresholds 
    table_maker(
        press_object=prs,
        layout_object=layout_blank,
        use_slide_object=True,
        slide_object=slide_temp,
        crosstab_override=filtered_thresholds,
        decimal_places=2,
        pos_horizontal=Inches(0.39),
        pos_vertical=Inches(4.97),
        width=Inches(6.15),
        height=Inches(1.28)
    )

    # Gráfico de ROI 
    chart_maker(
        press_object=prs,
        layout_object=layout_blank,
        use_slide_object=True,
        slide_object=slide_temp,
        crosstab_override=roi[roi.index == media].T,
        show_legend=False,
        show_category_axis=False,
        width=Inches(5.03),
        height=Inches(1.43),
        pos_horizontal=Inches(7.9),
        pos_vertical=Inches(1.87),
        chart_type=XL_CHART_TYPE.COLUMN_CLUSTERED,
        show_data_labels_value=XL_LABEL_POSITION.OUTSIDE_END
    )

    # Label gráfico ROI
    textbox_maker(
        slide_object=slide_temp,
        text='ROI',
        pos_horizontal=Inches(6.67),
        pos_vertical=Inches(1.87)
    )

    # Gráfico de efectividad
    chart_maker(
        press_object=prs,
        layout_object=layout_blank,
        use_slide_object=True,
        slide_object=slide_temp,
        crosstab_override=efectividad[efectividad.index == media].T,
        show_legend=False,
        show_category_axis=False,
        width=Inches(5.03),
        height=Inches(1.43),
        pos_horizontal=Inches(7.9),
        pos_vertical=Inches(3.32),
        chart_type=XL_CHART_TYPE.COLUMN_CLUSTERED,
        show_data_labels_value=XL_LABEL_POSITION.OUTSIDE_END
    )

    # Label gráfico efectividad
    textbox_maker(
        slide_object=slide_temp,
        text='Effectiveness',
        pos_horizontal=Inches(6.67),
        pos_vertical=Inches(3.32)
    )

    # Gráfico de costo por actvidad
    chart_maker(
        press_object=prs,
        layout_object=layout_blank,
        use_slide_object=True,
        slide_object=slide_temp,
        crosstab_override=cost_per_act[cost_per_act.index == media].T,
        show_legend=False,
        show_category_axis=True,
        width=Inches(5.03),
        height=Inches(1.86),
        pos_horizontal=Inches(7.9),
        pos_vertical=Inches(4.77),
        chart_type=XL_CHART_TYPE.COLUMN_CLUSTERED,
        show_data_labels_value=XL_LABEL_POSITION.OUTSIDE_END
    )

    # Label gráfico costo por actvidad
    textbox_maker(
        slide_object=slide_temp,
        text='CPA',
        pos_horizontal=Inches(6.67),
        pos_vertical=Inches(4.77)
    )


c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)
c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)
c:\Users\MonrroyF\Kantar\PersonalFernando - Documentos\Macros Analytics\Lift ROI Reporting Tool\functions\PPTX_CHARTER_KT_THEME.py:212: FutureWarning: Downcasting object dt

### Output 

In [22]:
prs.save('Output Automated Report.pptx')